# Agent Metrics: What You Get With Zero Extra Configuration

This notebook runs a travel agent doing its normal job — search a flight, check the weather, book it — against **real APIs**, and looks at the same run through two lenses: traditional logging, and the Strands SDK's built-in `EventLoopMetrics`.

No injected failures, no chaos effects. The point is to make the agent's *normal* behavior visible.

## The Tools

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `search_flights(origin, destination, departure_date)` | Searches real one-way fares via the Duffel **sandbox** API | Returns up to 5 offers sorted by price; remembers each `offer_id` so `book_flight` can validate it |
| `get_weather(city, target_date)` | Real daily forecast via Open-Meteo (no auth) | Only covers dates within ~16 days from today |
| `book_flight(offer_id, given_name, family_name, amount, currency)` | Writes a confirmed booking to a local SQLite ledger | No paid order is ever placed; `offer_id` must come from a prior `search_flights` call |


> **About the bookings database**: `book_flight` writes to a local **SQLite** file (`bookings.db`) — a stand-in booking system for these local demos. The bookings are simulated: no real airline order is ever placed. In the production demo (04), the same tool writes to **Amazon DynamoDB** instead, since the agent runs in the cloud.

## Docs this notebook is built from

- [Strands Agents: Logs](https://strandsagents.com/docs/user-guide/observability-evaluation/logs/) — `logging.getLogger("strands")`
- [Strands Agents: Metrics](https://strandsagents.com/docs/user-guide/observability-evaluation/metrics/) — `result.metrics.get_summary()`


In [1]:
import json
import logging
import os
from datetime import datetime, timedelta

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel  # OpenAI-compatible interface via Strands SDK

import travel_tools as T

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys "
        "and add it to a .env file."
    )

model = OpenAIModel(model_id="gpt-4o-mini")

SYSTEM_PROMPT = (
    "You are a travel assistant. Search flights, check the weather at the destination, "
    "and book the best option for the traveler without asking for confirmation. Be concise."
)

# Open-Meteo's forecast only covers the next ~16 days, so pick a near date at run time.
TRIP_DATE = (datetime.now() + timedelta(days=5)).strftime("%Y-%m-%d")
TRIP_PROMPT = (
    f"Book a one-way flight from JFK to MIA on {TRIP_DATE} for John Doe, "
    "and tell me if he'll need a jacket."
)

print("✅ Setup complete!")


✅ Setup complete!


## Lens 1 — traditional logging

`logging.getLogger("strands").setLevel(logging.DEBUG)` is the pattern from the [logs guide](https://strandsagents.com/docs/user-guide/observability-evaluation/logs/): "each module creates its own logger... all loggers are children of the 'strands' root logger."

Watch for lines like `tool_use=<...name': 'search_flights'...> | streaming` — they tell you a tool ran, but nothing about how many tokens the run cost.


In [ ]:
logging.getLogger("strands").setLevel(logging.DEBUG)
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
    force=True,
)

T.init_booking_db()
agent_logging = Agent(model=model, system_prompt=SYSTEM_PROMPT,
                       tools=[T.search_flights, T.get_weather, T.book_flight])
result_logging = agent_logging(TRIP_PROMPT)

logging.getLogger("strands").setLevel(logging.WARNING)


## Lens 2 — `result.metrics.get_summary()`

The [metrics guide](https://strandsagents.com/docs/user-guide/observability-evaluation/metrics/)'s own example: "a convenient `get_summary()` method... that gives you a comprehensive overview of your agent's performance in a single call." Zero extra install, zero extra config — it's already on the `AgentResult` the agent call returns.

Same trip, run fresh, so the ledger and the metrics both start clean.


In [3]:
T.init_booking_db()
agent_metrics = Agent(model=model, system_prompt=SYSTEM_PROMPT,
                       tools=[T.search_flights, T.get_weather, T.book_flight])
result_metrics = agent_metrics(TRIP_PROMPT)
summary = result_metrics.metrics.get_summary()





Tool #1: search_flights

Tool #2: get_weather

Tool #3: book_flight
John Doe's one-way flight from JFK to MIA has been booked with American Airlines for $89.58. The flight departs on July 27, 2026, at 8:07 PM and arrives at 11:02 PM. 

As for the weather in Miami, the temperature will be around 31.9°C (max) and 28.1°C (min). He likely won't need a jacket. 

Booking Reference: **BK-JQGDMD**.

In [4]:
print(json.dumps({
    "total_cycles": summary["total_cycles"],
    "total_duration_s": round(summary["total_duration"], 2),
    "accumulated_usage": summary["accumulated_usage"],
    "tool_usage": {
        name: {"call_count": data["execution_stats"]["call_count"],
               "success_count": data["execution_stats"]["success_count"],
               "average_time_s": round(data["execution_stats"]["average_time"], 3)}
        for name, data in summary.get("tool_usage", {}).items()
    },
}, indent=2))

{
  "total_cycles": 3,
  "total_duration_s": 7.2,
  "accumulated_usage": {
    "inputTokens": 2531,
    "outputTokens": 225,
    "totalTokens": 2756
  },
  "tool_usage": {
    "search_flights": {
      "call_count": 1,
      "success_count": 1,
      "average_time_s": 0.605
    },
    "get_weather": {
      "call_count": 1,
      "success_count": 1,
      "average_time_s": 0.817
    },
    "book_flight": {
      "call_count": 1,
      "success_count": 1,
      "average_time_s": 0.006
    }
  }
}


## Ground truth

The metrics summary tells you what the agent *did*. The SQLite ledger tells you what actually *persisted* — independent of anything the agent said in its final reply.


In [5]:
print(json.dumps(T.query_booked_offers(), indent=2))


[
  {
    "booking_reference": "BK-JQGDMD",
    "offer_id": "off_0000B8c9yTex5B3GJQgDMD",
    "passenger": "John Doe",
    "amount": "89.58",
    "currency": "USD"
  }
]


## Key takeaways

- **Logging** tells you *that* something happened (a tool was called, a response was generated).
- **`result.metrics.get_summary()`** tells you *how much* it cost: cycle count, token usage, and per-tool timing — with no install beyond `strands-agents`, no OpenTelemetry setup.
- **Ground truth** (the SQLite ledger here) is the only source that can't be fooled by what the agent claims in its reply.

`accumulated_metrics.latencyMs` reads `0` for the OpenAI provider — confirmed in the installed SDK source (`strands/models/openai.py`) as a `# TODO` gap in the provider's streaming-metrics code, not something this notebook works around. `accumulated_usage` (token counts) is accurate for every provider and is the number this notebook relies on.

## References

- [Strands Agents: Observability overview](https://strandsagents.com/docs/user-guide/observability-evaluation/observability/) · [Metrics](https://strandsagents.com/docs/user-guide/observability-evaluation/metrics/) · [Logs](https://strandsagents.com/docs/user-guide/observability-evaluation/logs/)
- Tools adapted from [Ricardo Ceci's `curso-strands-agentcore-2026`](https://github.com/ricardoceci/curso-strands-agentcore-2026)
- Next: [02 - OpenTelemetry Traces](../02-opentelemetry-traces/)
